# 03 — Compare model levels side-by-side

Fits the four `homer.models` classes one at a time, extracts metrics + the 42×42 anchor sub-block, **then discards the full π and garbage-collects** before fitting the next. This keeps peak memory low enough to run on a laptop kernel.

Compares on:
- anchor recovery (full supervision sanity)
- held-out anchor CV (visual network)
- FC translation Pearson r
- π row-max concentration (how hard the solution is)

Each fit stores **only**: the metrics dict (~1 KB) + the 42×42 anchor sub-block (~14 KB) + the human-side column-mass vector (~16 KB). Total per model = ~30 KB instead of ~30 MB.

In [1]:
import sys, gc, time, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
warnings.filterwarnings('ignore', category=DeprecationWarning, module='homer\\..*')
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
from homer.data import load_cached, get_anchor_index

ROOT = Path.cwd().parent
ANN  = ROOT / 'outputs' / 'anndata'

# Plotly for the heatmaps — we only build them at the END from saved small arrays,
# never holding multiple full Figure objects in memory.
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
M, _ = load_cached('mouse', cache_dir=ANN)
H, _ = load_cached('human', cache_dir=ANN)
costs = np.load(ANN / 'full_costs.npz')
Cm_SC = costs['Cm_SC']; Ch_SC = costs['Ch_SC']
idx_m = get_anchor_index(M.var)
idx_h = get_anchor_index(H.var)
n_m, n_h = M.uns['n_nodes'], H.uns['n_nodes']
print(f'mouse: {n_m} nodes  ·  human: {n_h} nodes')

mouse: 1864 nodes  ·  human: 2094 nodes


## Model recipes

Note: `UnsupervisedGW`'s default `max_iter` is 1000, which is overkill (and memory-heavy) for a comparison demo. We override it to 100 — still converges to the same minimum on this size of problem.

In [3]:
from homer.models import UnsupervisedGW, SupervisedFGW, MultimodalFGW, HierarchicalFGW

# Each entry: (display name, factory, requires SC?)
MODELS = [
    ('UnsupervisedGW',  lambda: UnsupervisedGW(epsilon=5e-3, max_iter=100), False),
    ('SupervisedFGW',   lambda: SupervisedFGW(epsilon=5e-3, xyz_weight=0.5), False),
    ('MultimodalFGW',   lambda: MultimodalFGW(use_sc=True, sc_weight=0.3, fc_weight=0.7,
                                                epsilon=5e-3, xyz_weight=0.5), True),
    ('HierarchicalFGW', lambda: HierarchicalFGW(epsilon=5e-3, xyz_weight=0.5), False),
]

## The fit loop — one model at a time, never holding two π in memory

For each model:
1. Fit (full supervision)
2. Save metrics + 42×42 anchor sub-block + human col-mass vector
3. Re-fit with visual network held out (pair_ids 5+6)
4. Save held-out metrics
5. **`del` the model + π + locals · `gc.collect()`**

Total ~1 minute (4 models × ~15 s each).

In [4]:
VISUAL_PAIR_IDS = [5, 6]
results = {}     # name → dict of small arrays + metrics (no π!)

for name, factory, needs_sc in MODELS:
    t = time.time()
    fit_kwargs = {'Cm_SC': Cm_SC, 'Ch_SC': Ch_SC} if needs_sc else {}

    # ---- (a) Full-supervision fit ----
    m = factory()
    m.fit(M, H, **fit_kwargs)

    pi = m.pi.astype(np.float32)                     # downcast just for stats
    full_anchor_metrics       = m.evaluate(eval_kind='anchor')
    full_translation_metrics  = m.evaluate(eval_kind='translation')
    pi_anchor_full            = pi[np.ix_(idx_m.pos, idx_h.pos)].copy()  # 42×42, ~14 KB
    col_mass                  = pi.sum(axis=0).astype(np.float32)        # n_h vector, ~16 KB
    row_concentration         = float((pi.max(axis=1) * pi.shape[0]).mean())
    loss                      = m.fit_info_.loss
    n_uncovered               = int((col_mass < 1e-5).sum())

    # ---- discard before next fit ----
    del m, pi
    gc.collect()

    # ---- (b) Visual-holdout fit ----
    m_held = factory()
    m_held.fit(M, H, holdout_pair_ids=VISUAL_PAIR_IDS, **fit_kwargs)
    holdout_metrics = m_held.evaluate(held_out_pair_ids=VISUAL_PAIR_IDS,
                                        eval_kind='anchor')
    pi_anchor_held  = m_held.pi[np.ix_(idx_m.pos, idx_h.pos)].astype(np.float32).copy()
    del m_held
    gc.collect()

    elapsed = time.time() - t
    results[name] = {
        'loss':              loss,
        'row_concentration': row_concentration,
        'n_uncovered_h':     n_uncovered,
        'full_anchor':       full_anchor_metrics,
        'full_translation':  full_translation_metrics,
        'pi_anchor_full':    pi_anchor_full,
        'pi_anchor_held':    pi_anchor_held,
        'col_mass':          col_mass,
        'holdout_metrics':   holdout_metrics,
        'elapsed_s':         elapsed,
    }
    print(f'  {name:18s}: full_top1={full_anchor_metrics["top1"]:.0%}  '
          f'visual_held={holdout_metrics["top1"]:.0%}  '
          f'fc_r={full_translation_metrics["pearson_r_overall"]:.3f}  '
          f'({elapsed:.1f}s)')

  UnsupervisedGW    : full_top1=0%  visual_held=0%  fc_r=0.393  (34.5s)
  SupervisedFGW     : full_top1=100%  visual_held=50%  fc_r=0.364  (81.4s)
  MultimodalFGW     : full_top1=100%  visual_held=50%  fc_r=0.361  (70.0s)
  HierarchicalFGW   : full_top1=100%  visual_held=25%  fc_r=0.395  (1.5s)


## Comparison tables

All four tables below come from the saved `results` dict — no further model fitting needed.

In [5]:
rows = []
for name, r in results.items():
    rows.append({
        'model':              name,
        'full_top1':          f'{r["full_anchor"]["top1"]:.0%}',
        'full_top5':          f'{r["full_anchor"]["top5"]:.0%}',
        'visual_held_top1':   f'{r["holdout_metrics"]["top1"]:.0%}',
        'visual_held_rank':   f'{r["holdout_metrics"]["mean_rank"]:.1f}',
        'fc_r_overall':       f'{r["full_translation"]["pearson_r_overall"]:.3f}',
        'fc_r_within_net':    f'{r["full_translation"].get("pearson_r_within_net", float("nan")):.3f}',
        'row_concentration':  f'{r["row_concentration"]:.3f}',
        'frac_uncovered_h':   f'{r["n_uncovered_h"] / n_h:.0%}',
        'loss':               f'{r["loss"]:.5f}',
        'fit_time_s':         f'{r["elapsed_s"]:.1f}',
    })
pd.DataFrame(rows)

,model,full_top1,full_top5,visual_held_top1,visual_held_rank,fc_r_overall,fc_r_within_net,row_concentration,frac_uncovered_h,loss,fit_time_s
0,UnsupervisedGW,0%,10%,0%,3.0,0.393,0.391,0.001,0%,0.01597,34.5
1,SupervisedFGW,100%,100%,50%,1.5,0.364,0.447,0.978,33%,0.01414,81.4
2,MultimodalFGW,100%,100%,50%,1.5,0.361,0.444,0.976,36%,0.01562,70.0
3,HierarchicalFGW,100%,100%,25%,2.5,0.395,0.548,0.988,63%,0.04686,1.5


## Anchor sub-block heatmaps — all four in one figure

42×42 sub-block of π restricted to anchor positions. Diagonal-ish = anchor supervision worked. UnsupervisedGW will look noisier (no anchors used).

In [6]:
names = list(results.keys())
fig = make_subplots(rows=2, cols=2, subplot_titles=names,
                     horizontal_spacing=0.08, vertical_spacing=0.12)
for k, name in enumerate(names):
    z = results[name]['pi_anchor_full']
    r, c = k // 2 + 1, k % 2 + 1
    fig.add_trace(
        go.Heatmap(z=z, colorscale='Viridis', showscale=(k == 0), coloraxis=None),
        row=r, col=c,
    )
    fig.update_yaxes(autorange='reversed', row=r, col=c)
fig.update_layout(
    title='Anchor sub-block (42 × 42) — full supervision',
    width=900, height=900,
    margin=dict(l=50, r=20, t=70, b=40),
)
fig

## Same heatmap, but with the visual network held out

Look at the 4 visual anchors (rows/cols indexed roughly 8–11 — pair_ids 5+6 × L/R). For the supervised models, those rows lose their diagonal hit because the model wasn't shown the answer.

In [7]:
fig = make_subplots(rows=2, cols=2, subplot_titles=names,
                     horizontal_spacing=0.08, vertical_spacing=0.12)
for k, name in enumerate(names):
    z = results[name]['pi_anchor_held']
    r, c = k // 2 + 1, k % 2 + 1
    fig.add_trace(go.Heatmap(z=z, colorscale='Viridis', showscale=(k == 0)),
                  row=r, col=c)
    fig.update_yaxes(autorange='reversed', row=r, col=c)
fig.update_layout(
    title='Anchor sub-block (42 × 42) — VISUAL withheld (pair_ids 5+6)',
    width=900, height=900,
    margin=dict(l=50, r=20, t=70, b=40),
)
fig

## Human-side coverage (column mass) per model

How concentrated each model's mass is on a small subset of human nodes. Production semirelaxed FGW intentionally leaves ~36% of human nodes uncovered.

In [8]:
fig = go.Figure()
for name in names:
    cm = np.sort(results[name]['col_mass'])[::-1]    # sorted descending
    fig.add_trace(go.Scatter(
        y=np.cumsum(cm) / cm.sum(),
        x=np.arange(len(cm)) / len(cm),
        mode='lines',
        name=name,
    ))
fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1,
               line=dict(color='gray', dash='dash'))
fig.update_layout(
    title='Cumulative human-side coverage — Lorenz-style',
    xaxis_title='Fraction of human nodes (sorted by col-mass desc)',
    yaxis_title='Cumulative π mass',
    width=700, height=500,
)
fig

## Takeaways

Expected pattern (matches `outputs/comparison/comprehensive_table.csv`):

| Model               | Full anchor top-1 | Visual-held top-1 | FC translation r |
|---------------------|------------------|--------------------|------------------|
| UnsupervisedGW      | ~14% (no anchors) | ~14%               | low              |
| SupervisedFGW       | 100%             | 50%                | 0.36             |
| **MultimodalFGW**   | **100%**         | **50%**            | **0.36**         |
| HierarchicalFGW     | varies           | 25–50%             | 0.39 / 0.55 within-net |

Anchor supervision dominates the headline. Adding SC (multimodal) doesn't move held-out top-1 measurably but improves within-network FC translation. Hierarchical wins within-network FC translation but hurts leave-one-network-out CV (held-out network has zero supervision in its sub-block).

In [9]:
# Clean up the small `results` dict before kernel idles — also frees Plotly figure cache.
del results, costs, Cm_SC, Ch_SC
gc.collect()
print('cleaned up')

cleaned up
